# Grouped Cross-Validation for Medical AI — trustcv Complete Showcase

> **Package:** `trustcv` v1.0.7 · Developed at SMAILE, Karolinska Institutet  
> **Repo:** https://github.com/ki-smile/trustcv

This notebook uses **only real trustcv APIs** read directly from the GitHub source.
It covers every grouped CV method with a realistic multi-site clinical dataset.

### APIs demonstrated
| API | Module | Purpose |
|-----|--------|---------|
| `TrustCV` / `TrustCVValidator` | `trustcv` | High-level CV with leakage checking |
| `UniversalCVRunner` | `trustcv` | Framework-agnostic CV loop with callbacks |
| `CVResults` | `trustcv` | Structured results container |
| `ClinicalMetrics` | `trustcv` | Sensitivity/Specificity/PPV/NPV/LR with Wilson CIs |
| `oob_clinical_metrics` | `trustcv.metrics` | OOB pooled clinical metrics from CVResults |
| `DataLeakageChecker` | `trustcv` | Patient-level leakage detection |
| `LeakageDetectionCallback` | `trustcv` | Real-time leakage alert during CV |
| `ClassDistributionLogger` | `trustcv` | Log class balance each fold |
| `results.dashboard()` | `trustcv` | Interactive Plotly dashboard |
| `results.summary()` | `trustcv` | Text summary |
| `generate_synthetic_ehr` | `trustcv.datasets` | Built-in medical dataset generator |

### Grouped CV methods
| Method | Use case |
|--------|---------|
| `GroupKFoldMedical` | Standard patient-level K-Fold |
| `StratifiedGroupKFold` | Patient K-Fold + class balance preservation |
| `RepeatedGroupKFold` | K-Fold × N repeats — reduces variance |
| `HierarchicalGroupKFold` | Hospital → patient hierarchy (multi-site) |
| `LeaveOneGroupOut` | Leave-one-patient-out |
| `LeavePGroupsOut` | Leave-P-patients-out |
---


## Cell 1 — Install trustcv


In [ ]:
# Uncomment and run this cell first in Colab, then restart the runtime
!pip install "git+https://github.com/ki-smile/trustcv.git" \
#     --force-reinstall --no-cache-dir -q
!pip install plotly -q

import importlib.metadata
print(f"trustcv version: {importlib.metadata.version('trustcv')}")


  Cloning https://github.com/ki-smile/trustcv.git to /tmp/pip-req-build-mqg1qy1i
  Running command git clone --filter=blob:none --quiet https://github.com/ki-smile/trustcv.git /tmp/pip-req-build-mqg1qy1i
  Resolved https://github.com/ki-smile/trustcv.git to commit 34ecae39f1509d455b162e9a8141335f969296eb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for trustcv: filename=trustcv-1.0.7-py3-none-any.whl size=149359 sha256=b028dd9fedbe2665bbbfbfdf1a3333d721dc0aaf2b4f7b9c66e95eb8d2ad92d9
  Stored in directory: /tmp/pip-ephem-wheel-cache-58g0lat0/wheels/50/50/a0/a8f0fce3cd30d6e568a9aabfa60d4cbe04f9f33c9dd933e0c3
Successfully built trustcv
  Attempting uninstall: trustcv
    Found existing installation: trustcv 1.0.5
    Uninstalling trustcv-1.0.5:
      Successfully uninstalled trustcv-1.0.5

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install 

## Cell 2 — Imports


In [2]:
import sys
from pathlib import Path

# Ensure local repo version of trustcv is imported when running outside Colab.
_REPO_TRUSTCV = Path('/home/amir/Umea_Desctop/AAA_test/p4_CV/P4_paper/Repo/trustcv26012026')
if _REPO_TRUSTCV.exists():
    if str(_REPO_TRUSTCV) not in sys.path:
        sys.path.insert(0, str(_REPO_TRUSTCV))
    for _mod in list(sys.modules):
        if _mod == 'trustcv' or _mod.startswith('trustcv.'):
            del sys.modules[_mod]

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, HTML

# ── trustcv top-level exports ────────────────────────────────────────────────
from trustcv import (
    # Validators
    TrustCV,
    TrustCVValidator,

    # Framework-agnostic runner
    UniversalCVRunner,
    CVResults,

    # Callbacks (used inside UniversalCVRunner)
    LeakageDetectionCallback,
    ClassDistributionLogger,

    # Clinical metrics
    ClinicalMetrics,

    # Leakage checker
    DataLeakageChecker,

    # Grouped CV splitters
    GroupKFoldMedical,
    StratifiedGroupKFold,
    LeaveOneGroupOut,
    LeavePGroupsOut,
    RepeatedGroupKFold,
    HierarchicalGroupKFold,
    NestedGroupedCV,

    # Built-in dataset generator
    generate_synthetic_ehr,
)

# ── oob_clinical_metrics lives in trustcv.metrics ────────────────────────────
from trustcv.metrics import oob_clinical_metrics

# ── sklearn helpers ───────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.base import clone

print("All imports OK ✓")


All imports OK ✓


## Cell 3 — Clinical dataset

We use `generate_synthetic_ehr` — the **built-in trustcv dataset generator** —
to create a synthetic multi-site EHR dataset simulating a sepsis early-warning study:

- **600 visits** across **~88 unique patients** in **5 hospitals**
- **20 clinical features** (labs, vitals, demographics)
- **Binary outcome**: sepsis (1) vs. no sepsis (0), prevalence ≈ 30%
- Patient IDs encoded to integers for grouped splitters


In [3]:
# ── Generate dataset using trustcv's built-in generator ──────────────────────
ehr = generate_synthetic_ehr(
    n_samples  = 600,
    n_patients = 100,
    n_features = 20,
    prevalence = 0.30,
    random_state = 42,
)

X           = np.array(ehr["X"])
y           = np.array(ehr["y"])
# patient_ids come as strings (e.g. "EHR_00042") — encode to ints
patient_ids = LabelEncoder().fit_transform(ehr["patient_ids"])

# Assign each patient to one of 5 hospitals
np.random.seed(42)
n_patients      = len(np.unique(patient_ids))
patient_hospital = np.random.randint(0, 5, n_patients)
hospital_ids    = patient_hospital[patient_ids]

PREVALENCE = y.mean()

print(f"Samples          : {len(X)}")
print(f"Features         : {X.shape[1]}")
print(f"Unique patients  : {n_patients}")
print(f"Unique hospitals : {len(np.unique(hospital_ids))}")
print(f"Class balance    : Sepsis={y.sum()} ({PREVALENCE*100:.1f}%)  "
      f"No-sepsis={len(y)-y.sum()} ({(1-PREVALENCE)*100:.1f}%)")

# Sample data view
df_view = pd.DataFrame(X[:5], columns=[f"feature_{i}" for i in range(X.shape[1])])
df_view.insert(0, "patient_id", patient_ids[:5])
df_view.insert(1, "hospital",   hospital_ids[:5])
df_view["sepsis"] = y[:5]
display(df_view.round(3))


Samples          : 600
Features         : 20
Unique patients  : 88
Unique hospitals : 5
Class balance    : Sepsis=192 (32.0%)  No-sepsis=408 (68.0%)


,patient_id,hospital,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,...,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,sepsis
0,1,4,-1.666,-2.672,-4.287,-2.839,-4.691,1.442,0.365,-1.969,...,3.653,-0.318,-3.662,-1.092,6.528,0.729,-0.035,3.511,0.557,0
1,55,0,-2.185,1.292,-2.908,3.769,-0.783,0.638,-0.087,2.731,...,3.271,5.346,-2.644,0.324,-14.123,0.220,0.566,-4.600,0.060,0
2,27,1,0.131,2.630,-1.779,1.113,2.084,0.240,-0.196,-0.683,...,0.135,-1.309,-1.680,0.035,-3.027,1.761,-2.131,-1.291,-0.803,1
3,44,1,1.621,3.478,4.623,0.736,0.095,-1.336,0.569,5.396,...,0.016,4.558,-2.025,-1.101,-1.618,-1.921,3.378,2.225,-0.617,0
4,78,3,1.702,-1.863,5.054,3.229,4.551,0.427,1.791,-1.826,...,-3.457,6.143,1.029,0.844,-2.978,0.731,1.943,-4.145,-0.442,1


## Cell 4 — Configure all grouped CV methods

All six grouped splitters are configured. Each uses **patient IDs or hospital IDs**
as the grouping variable, preventing any patient from appearing in both train and test.


In [4]:
# Model used across all methods
BASE_MODEL = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
)

# LOGO subset: first 25 patients (LOGO on all 88 = too slow for demo)
LOGO_MASK = patient_ids < 25

# ── Method registry ───────────────────────────────────────────────────────────
CV_CONFIGS = {
    "GroupKFoldMedical": {
        "splitter":    GroupKFoldMedical(n_splits=5, random_state=42),
        "groups":      patient_ids,
        "mask":        None,
        "color":       "#378ADD",
        "description": "5-fold, strict patient separation (standard choice)",
    },
    "StratifiedGroupKFold": {
        "splitter":    StratifiedGroupKFold(n_splits=5, random_state=42),
        "groups":      patient_ids,
        "mask":        None,
        "color":       "#1D9E75",
        "description": "5-fold + class balance per fold (imbalanced outcomes)",
    },
    "RepeatedGroupKFold": {
        "splitter":    RepeatedGroupKFold(n_splits=5, n_repeats=3, random_state=42),
        "groups":      patient_ids,
        "mask":        None,
        "color":       "#D85A30",
        "description": "5-fold × 3 repeats = 15 folds (lower variance estimate)",
    },
    "HierarchicalGroupKFold": {
        "splitter":    HierarchicalGroupKFold(n_splits=5, random_state=42),
        "groups":      hospital_ids,
        "mask":        None,
        "color":       "#7F77DD",
        "description": "Hospital-level 5-fold (multi-site trial evaluation)",
    },
    "LeaveOneGroupOut": {
        "splitter":    LeaveOneGroupOut(),
        "groups":      patient_ids[LOGO_MASK],
        "mask":        LOGO_MASK,
        "color":       "#D4537E",
        "description": "Leave-one-patient-out on 25 patients (small cohort)",
    },
   # "LeavePGroupsOut": {
   #     "splitter":    LeavePGroupsOut(n_groups=2),
   #     "groups":      patient_ids,
   #     "mask":        None,
   #     "color":       "#888780",
   #     "description": "Leave-2-patients-out (conservative evaluation)",
   # },
}

print(f"{'Method':<25}  {'Folds':>6}  Description")
print("-" * 80)
for name, cfg in CV_CONFIGS.items():
    mask = cfg["mask"]
    X_u = X[mask] if mask is not None else X
    y_u = y[mask] if mask is not None else y
    g_u = cfg["groups"]
    try:
        n_folds = len(list(cfg["splitter"].split(X_u, y_u, g_u)))
        # LeavePGroupsOut has too many folds — cap display
        n_disp = f"{min(n_folds, 9999):>6}" if n_folds < 9999 else " many"
        print(f"  {name:<25}  {n_disp}  {cfg['description']}")
    except Exception as e:
        print(f"  {name:<25}  ERROR: {e}")


Method                      Folds  Description
--------------------------------------------------------------------------------


  GroupKFoldMedical               5  5-fold, strict patient separation (standard choice)
  StratifiedGroupKFold            5  5-fold + class balance per fold (imbalanced outcomes)
  RepeatedGroupKFold             15  5-fold × 3 repeats = 15 folds (lower variance estimate)
  HierarchicalGroupKFold          5  Hospital-level 5-fold (multi-site trial evaluation)
  LeaveOneGroupOut               25  Leave-one-patient-out on 25 patients (small cohort)


## Cell 5 — `UniversalCVRunner` with callbacks

`UniversalCVRunner` is the trustcv framework-agnostic CV engine. We attach two callbacks:
- `LeakageDetectionCallback` — alerts if any patient leaks between train/test
- `ClassDistributionLogger` — prints class balance each fold

We run it for the 4 main methods (skipping LOGO and LeavePGroupsOut for speed).


In [ ]:
import time

runner_results = {}
RUN_METHODS = ["GroupKFoldMedical", "StratifiedGroupKFold",
               "RepeatedGroupKFold", "HierarchicalGroupKFold"]

for method_name in RUN_METHODS:
    cfg  = CV_CONFIGS[method_name]
    mask = cfg["mask"]
    X_u  = X[mask] if mask is not None else X
    y_u  = y[mask] if mask is not None else y

    runner = UniversalCVRunner(
        cv_splitter = cfg["splitter"],
        verbose     = 2,
    )
    callbacks = [
        LeakageDetectionCallback(data=(X_u, y_u), groups=cfg["groups"], verbose=1),
        ClassDistributionLogger(labels=y_u, verbose=0),
    ]

    print("=" * 60)
    print("  " + method_name)
    print("=" * 60)
    t0 = time.time()

    cv_res = runner.run(
        model     = clone(BASE_MODEL),
        data      = (X_u, y_u),
        groups    = cfg["groups"],
        metrics   = ["accuracy", "roc_auc", "f1", "precision", "recall"],
        callbacks = callbacks,
    )
    elapsed = time.time() - t0
    # 5. Inspect Results
    print("\n=== CV Results Summary ===")
    print(cv_res.summary())



    ms  = cv_res.mean_scores
    std = cv_res.std_scores

    print("  Folds    : " + str(len(cv_res.scores)))
    print("  Accuracy : " + f"{ms.get('accuracy',0):.3f} +/- {std.get('accuracy',0):.3f}")
    print("  ROC-AUC  : " + f"{ms.get('roc_auc',0):.3f} +/- {std.get('roc_auc',0):.3f}")
    print("  F1       : " + f"{ms.get('f1',0):.3f} +/- {std.get('f1',0):.3f}")
    print("  Time     : " + f"{elapsed:.1f}s")

    runner_results[method_name] = cv_res

print("\nUniversalCVRunner complete for all 4 methods ✓")


  GroupKFoldMedical
Using sklearn adapter for cross-validation
Starting 5-fold cross-validation

Fold 1:
  Training samples: 481
  Validation samples: 119
  Training groups: 70
  Validation groups: 18
Fold 1 completed
  score: 0.8403
  accuracy: 0.8403
  f1: 0.6984
  precision: 0.8148
  recall: 0.6111
  balanced_accuracy: 0.7754
  roc_auc: 0.8998

Fold 2:
  Training samples: 461
  Validation samples: 139
  Training groups: 70
  Validation groups: 18
Fold 2 completed
  score: 0.8129
  accuracy: 0.8129
  f1: 0.6667
  precision: 0.8387
  recall: 0.5532
  balanced_accuracy: 0.7494
  roc_auc: 0.8336

Fold 3:
  Training samples: 494
  Validation samples: 106
  Training groups: 70
  Validation groups: 18
Fold 3 completed
  score: 0.7925
  accuracy: 0.7925
  f1: 0.6207
  precision: 0.7500
  recall: 0.5294
  balanced_accuracy: 0.7230
  roc_auc: 0.8401

Fold 4:
  Training samples: 467
  Validation samples: 133
  Training groups: 71
  Validation groups: 17
Fold 4 completed
  score: 0.8195
  accur

## Cell 6 — `oob_clinical_metrics` — pooled OOB clinical performance

`oob_clinical_metrics(cv_res, y)` pools all fold test-set predictions from a `CVResults`
object and computes **full clinical metrics** (sensitivity, specificity, PPV, NPV,
likelihood ratios, diagnostic OR, Youden index) with **Wilson 95% confidence intervals**.

This is the recommended way to get a single aggregate clinical performance estimate
from a grouped CV run — no bootstrap loop needed.


In [ ]:
cm_obj = ClinicalMetrics(
    confidence_level = 0.95,
    prevalence       = PREVALENCE,   # use dataset prevalence for NNT/NNS
)

oob_results = {}

for method_name, cv_res in runner_results.items():
    mask = CV_CONFIGS[method_name]["mask"]
    y_u  = y[mask] if mask is not None else y

    oob = oob_clinical_metrics(
        results  = cv_res,
        y        = y_u,
        clinical = cm_obj,
    )
    oob_results[method_name] = oob

    print(f"\n── {method_name} ──")
    print(f"  Sensitivity  : {oob['sensitivity']:.3f}  "
          f"95% CI [{oob['sensitivity_ci'][0]:.3f}, {oob['sensitivity_ci'][1]:.3f}]")
    print(f"  Specificity  : {oob['specificity']:.3f}  "
          f"95% CI [{oob['specificity_ci'][0]:.3f}, {oob['specificity_ci'][1]:.3f}]")
    print(f"  PPV          : {oob['ppv']:.3f}  "
          f"95% CI [{oob['ppv_ci'][0]:.3f}, {oob['ppv_ci'][1]:.3f}]")
    print(f"  NPV          : {oob['npv']:.3f}  "
          f"95% CI [{oob['npv_ci'][0]:.3f}, {oob['npv_ci'][1]:.3f}]")
    print(f"  ROC-AUC      : {oob['auc_roc']:.3f}  "
          f"95% CI [{oob['auc_roc_ci'][0]:.3f}, {oob['auc_roc_ci'][1]:.3f}]")
    print(f"  Youden J     : {oob['youdens_index']:.3f}")
    print(f"  LR+          : {oob['lr_positive']:.2f}  → {ClinicalMetrics._interpret_lr_positive(oob['lr_positive'])}")
    print(f"  LR-          : {oob['lr_negative']:.2f}  → {ClinicalMetrics._interpret_lr_negative(oob['lr_negative'])}")
    if oob.get("nnt"):
        print(f"  NNT          : {oob['nnt']:.1f}")
    if oob.get("nns"):
        print(f"  NNS          : {oob['nns']:.1f}")
    cm_data = oob["confusion_matrix"]
    print(f"  Confusion    : TP={cm_data['true_positives']}  TN={cm_data['true_negatives']}  "
          f"FP={cm_data['false_positives']}  FN={cm_data['false_negatives']}")


## Cell 7 — `ClinicalMetrics.format_report` — full clinical report

`format_report()` formats the dictionary returned by `oob_clinical_metrics` into a
structured clinical report suitable for regulatory documentation.


In [ ]:
# Print the full clinical report for the best-performing method
best_method = max(oob_results, key=lambda n: oob_results[n]["auc_roc"])
print(f"Full clinical report for: {best_method}\n")
report = cm_obj.format_report(oob_results[best_method])
print(report)


## Cell 8 — `DataLeakageChecker` — explicit per-fold leakage audit

Beyond the `LeakageDetectionCallback` used during CV, we can run `DataLeakageChecker`
directly on any train/test split to get a detailed `LeakageReport` with severity ratings.


In [ ]:
checker = DataLeakageChecker(verbose=False)

for method_name in ["GroupKFoldMedical", "StratifiedGroupKFold", "HierarchicalGroupKFold"]:
    cfg     = CV_CONFIGS[method_name]
    mask    = cfg["mask"]
    X_u     = X[mask] if mask is not None else X
    y_u     = y[mask] if mask is not None else y
    g_u     = cfg["groups"]
    indices = runner_results[method_name].indices

    print(f"\n── {method_name} ──")
    all_pass = True

    for fold_i, (tr_idx, te_idx) in enumerate(indices[:3]):   # check first 3 folds
        report = checker.check_cv_splits(
            X_train           = X_u[tr_idx],
            X_test            = X_u[te_idx],
            y_train           = y_u[tr_idx],
            y_test            = y_u[te_idx],
            patient_ids_train = g_u[tr_idx],
            patient_ids_test  = g_u[te_idx],
        )
        icon   = "✓" if not report.has_leakage else "✗"
        status = "PASS" if not report.has_leakage else "FAIL"
        overlap = report.details.get("patient_leakage", {}).get("overlap_count", 0)
        print(f"  Fold {fold_i+1}: {icon} {status}  severity={report.severity:<8}  "
              f"patient_overlap={overlap}  "
              f"leakage_types={report.leakage_types if report.leakage_types else 'none'}")
        if report.has_leakage:
            all_pass = False

    print(f"  → {'All checked folds passed ✓' if all_pass else 'LEAKAGE DETECTED ✗'}")


## Cell 9 — `TrustCV` high-level validator with `results.summary()`

`TrustCV` (alias for `TrustCVValidator`) provides the highest-level interface.
Pass `method='group_kfold'` and `groups=patient_ids` for patient-level CV.
The result has `.summary()` and `.dashboard()` built in.


In [ ]:
trustcv_results = {}

method_specs = [
    ("group_kfold", "GroupKFoldMedical", None),
    ("stratified_group_kfold", "StratifiedGroupKFold", None),
    (
        "repeated_group_kfold",
        "RepeatedGroupKFold",
        RepeatedGroupKFold(n_splits=5, n_repeats=2, random_state=42),
    ),
    (
        "hierarchical_group_kfold",
        "HierarchicalGroupKFold",
        HierarchicalGroupKFold(n_splits=5, hierarchy_level="patient", random_state=42),
    ),
]

for method_str, splitter_key, fallback_cv in method_specs:
    print(f"\n── TrustCV method='{method_str}' ──")

    validator_kwargs = dict(
        method=method_str,
        n_splits=5,
        random_state=42,
        check_leakage=True,
        check_balance=True,
    )
    if method_str == "repeated_group_kfold":
        validator_kwargs["n_repeats"] = 2
    if method_str == "hierarchical_group_kfold":
        validator_kwargs["hierarchy_level"] = "patient"

    try:
        validator = TrustCV(**validator_kwargs)
        vr = validator.validate(
            model=clone(BASE_MODEL),
            X=X,
            y=y,
            groups=patient_ids,
        )
    except ValueError as e:
        # Backward compatibility for older trustcv versions that don't parse newer method names.
        if "Unknown method" in str(e) and fallback_cv is not None:
            print("Method alias unavailable in current runtime; using explicit splitter fallback.")
            validator = TrustCV(
                method="group_kfold",
                n_splits=5,
                random_state=42,
                check_leakage=True,
                check_balance=True,
            )
            vr = validator.validate(
                model=clone(BASE_MODEL),
                X=X,
                y=y,
                groups=patient_ids,
                cv=fallback_cv,
            )
        else:
            raise

    trustcv_results[splitter_key] = vr
    print(vr.summary())


## Cell 10 — `results.dashboard()` — interactive Plotly dashboard

Call `results.dashboard()` on any `ValidationResult` returned by `TrustCV.validate()`.
This opens 6 interactive Plotly charts: metric bars, fold lines, CI ranges, heatmap,
integrity checks table, and run summary.


In [ ]:
# Dashboard for the GroupKFold result
print("Opening dashboard for GroupKFoldMedical...")
trustcv_results["GroupKFoldMedical"].dashboard(
    title="Sepsis Early-Warning — GroupKFoldMedical"
)


In [ ]:
# Dashboard for StratifiedGroupKFold  
print("Opening dashboard for StratifiedGroupKFold...")
trustcv_results["StratifiedGroupKFold"].dashboard(
    title="Sepsis Early-Warning — StratifiedGroupKFold"
)


## Cell 11 — Comparison: all methods side-by-side

Custom Plotly figures comparing all 4 methods across accuracy, ROC-AUC,
sensitivity, and specificity using the `oob_clinical_metrics` pooled estimates.


In [ ]:
METHODS     = list(oob_results.keys())
COLORS      = [CV_CONFIGS[n]["color"] for n in METHODS]
BG, CARD    = "#F8F8F6", "#FFFFFF"
LINE_COL    = "#E0DED8"
FONT        = dict(family="monospace, sans-serif", color="#2C2C2A")

def _layout(subtitle, height=400):
    return dict(
        title=dict(
            text=f"<b>trustcv — Grouped CV Comparison</b>  ·  {subtitle}",
            font=dict(size=14, color="#2C2C2A"), x=0.01),
        paper_bgcolor=BG, plot_bgcolor=CARD, font=FONT,
        height=height, margin=dict(l=60, r=40, t=55, b=70),
    )

# ── Fig A: OOB clinical metrics grouped bar ───────────────────────────────────
CLIN_KEYS = ["sensitivity","specificity","ppv","npv","auc_roc"]
CLIN_DISP = ["Sensitivity","Specificity","PPV","NPV","ROC-AUC"]

fig_a = go.Figure()
for name, color in zip(METHODS, COLORS):
    oob  = oob_results[name]
    vals = [oob[k]*100 for k in CLIN_KEYS]
    # CI error bars (upper only for readability)
    ci_hi = [(oob.get(k+"_ci",(oob[k],oob[k]))[1] - oob[k])*100 for k in CLIN_KEYS]
    ci_lo = [(oob[k] - oob.get(k+"_ci",(oob[k],oob[k]))[0])*100 for k in CLIN_KEYS]
    fig_a.add_trace(go.Bar(
        x=CLIN_DISP, y=vals, name=name,
        marker_color=color, marker_line_width=0,
        error_y=dict(type="data", symmetric=False,
                     array=ci_hi, arrayminus=ci_lo,
                     color="#5F5E5A", thickness=1.2, width=4),
        offsetgroup=name, width=0.18,
        hovertemplate=f"<b>{name}</b><br>%{{x}}: %{{y:.1f}}% ± CI<extra></extra>",
    ))
fig_a.update_layout(
    **_layout("OOB clinical metrics — all grouped CV methods", 450),
    barmode="group",
    yaxis=dict(title="Score (%)", range=[40,108], gridcolor=LINE_COL, zeroline=False),
    xaxis=dict(gridcolor="rgba(0,0,0,0)"),
    legend=dict(orientation="h", y=-0.25, x=0, font=dict(size=11)),
    showlegend=True,
)
fig_a.show()


In [ ]:
# ── Fig B: Per-fold AUC line chart ────────────────────────────────────────────
fig_b = go.Figure()
symbols = ["circle","square","diamond","cross"]

for i, (name, color, sym) in enumerate(zip(METHODS, COLORS, symbols)):
    cv_res = runner_results[name]
    fold_aucs = [s.get("roc_auc", 0)*100 for s in cv_res.scores]
    fold_labels = [f"F{j+1}" for j in range(len(fold_aucs))]
    fig_b.add_trace(go.Scatter(
        x=fold_labels, y=fold_aucs,
        mode="lines+markers", name=name,
        line=dict(color=color, width=2),
        marker=dict(symbol=sym, size=9, color=color,
                    line=dict(width=1.5, color="white")),
        hovertemplate=f"<b>{name}</b> %{{x}}: %{{y:.1f}}%<extra></extra>",
    ))
    fig_b.add_hline(
        y=np.mean(fold_aucs),
        line_dash="dot", line_color=color, line_width=1, opacity=0.4,
    )

fig_b.update_layout(
    **_layout("Per-fold ROC-AUC  (dashed = method mean)", 420),
    yaxis=dict(title="ROC-AUC (%)", range=[50,105], gridcolor=LINE_COL, zeroline=False),
    xaxis=dict(title="Fold index", gridcolor="rgba(0,0,0,0)"),
    legend=dict(orientation="h", y=-0.25, x=0, font=dict(size=11)),
    showlegend=True,
)
fig_b.show()


In [ ]:
# ── Fig C: Clinical metrics heatmap ──────────────────────────────────────────
fig_c = go.Figure(go.Heatmap(
    z=[[oob_results[m][k]*100 for m in METHODS] for k in CLIN_KEYS],
    x=[n.replace("GroupKFoldMedical","GKFold").replace("StratifiedGroupKFold","StratGKF")
        .replace("RepeatedGroupKFold","RepGKF").replace("HierarchicalGroupKFold","HierGKF")
       for n in METHODS],
    y=CLIN_DISP,
    colorscale=[[0,"#EEF4FB"],[0.5,"#85B7EB"],[1,"#0C447C"]],
    zmin=50, zmax=100,
    texttemplate="%{z:.1f}%",
    textfont=dict(size=13),
    showscale=False,
    hovertemplate="<b>%{y}</b> — %{x}<br>%{z:.1f}%<extra></extra>",
))
fig_c.update_layout(
    **_layout("OOB clinical metrics heatmap", 320),
    xaxis=dict(side="top", gridcolor="rgba(0,0,0,0)"),
    yaxis=dict(gridcolor="rgba(0,0,0,0)"),
    showlegend=False,
)
fig_c.show()


In [ ]:
# ── Fig D: Youden index + LR+ comparison ─────────────────────────────────────
fig_d = go.Figure()
youden_vals = [oob_results[n]["youdens_index"] for n in METHODS]
lrp_vals    = [min(oob_results[n]["lr_positive"], 20) for n in METHODS]  # cap at 20

short = [n.replace("GroupKFoldMedical","GKFold").replace("StratifiedGroupKFold","StratGKF")
          .replace("RepeatedGroupKFold","RepGKF").replace("HierarchicalGroupKFold","HierGKF")
         for n in METHODS]

fig_d.add_trace(go.Bar(
    x=short, y=youden_vals, name="Youden J",
    marker_color=COLORS, marker_line_width=0,
    text=[f"{v:.3f}" for v in youden_vals], textposition="outside",
    hovertemplate="<b>%{x}</b><br>Youden J: %{y:.3f}<extra></extra>",
))
fig_d.update_layout(
    **_layout("Youden's Index per method  (higher = better)", 380),
    yaxis=dict(title="Youden J (= Sensitivity + Specificity − 1)",
               range=[0, 1.1], gridcolor=LINE_COL, zeroline=False),
    xaxis=dict(gridcolor="rgba(0,0,0,0)"),
    showlegend=False,
)
fig_d.show()


## Cell 12 — Final summary table


In [ ]:
import pandas as pd

rows = []
for name in METHODS:
    oob = oob_results[name]
    ms  = runner_results[name].mean_scores
    std = runner_results[name].std_scores
    lo_s, hi_s = oob["sensitivity_ci"]
    lo_sp, hi_sp = oob["specificity_ci"]
    lo_a, hi_a = oob["auc_roc_ci"]
    rows.append({
        "Method":          name,
        "Folds":           len(runner_results[name].scores),
        "Acc (CV)":        f"{ms.get('accuracy',0)*100:.1f}% +/-{std.get('accuracy',0)*100:.1f}",
        "AUC (CV)":        f"{ms.get('roc_auc',0)*100:.1f}% +/-{std.get('roc_auc',0)*100:.1f}",
        "Sens OOB":        f"{oob['sensitivity']*100:.1f}% [{lo_s*100:.1f}-{hi_s*100:.1f}]",
        "Spec OOB":        f"{oob['specificity']*100:.1f}% [{lo_sp*100:.1f}-{hi_sp*100:.1f}]",
        "PPV OOB":         f"{oob['ppv']*100:.1f}%",
        "NPV OOB":         f"{oob['npv']*100:.1f}%",
        "AUC OOB":         f"{oob['auc_roc']*100:.1f}% [{lo_a*100:.1f}-{hi_a*100:.1f}]",
        "Youden J":        f"{oob['youdens_index']:.3f}",
        "LR+":             f"{min(oob['lr_positive'], 99.9):.2f}",
        "LR-":             f"{oob['lr_negative']:.2f}",
    })

df_final = pd.DataFrame(rows)
display(df_final.style
    .hide(axis="index")
    .set_caption("Grouped CV methods — OOB clinical performance (trustcv)")
    .set_table_styles([
        {"selector":"caption","props":[("font-size","15px"),("font-weight","700"),
                                        ("text-align","left"),("padding-bottom","8px")]},
        {"selector":"th","props":[("background","#378ADD"),("color","white"),
                                   ("padding","8px 14px"),("font-size","12px")]},
        {"selector":"tr:nth-child(even)","props":[("background","#F8F8F6")]},
    ])
    .set_properties(**{"font-size":"12px","text-align":"center"})
)
best = max(oob_results, key=lambda n: oob_results[n]["auc_roc"])
print("Done — all cells executed ✓")
print("Best AUC method: " + best + " (" + f"{oob_results[best]['auc_roc']*100:.1f}%" + ")")
